Install Conda Colab as well as dependencies for OpenMM.

Need to rerun every time this notebook is connected as all data is wiped. Don't mind the crash message when condacolab is installed, it will still work

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...


In [1]:
!mamba install -y -c conda-forge openmm openmmforcefields parmed pdb2pqr pdbfixer
!python -m openmm.testInstallation

[+] 0.0s
[+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   9%
conda-forge/noarch    16%[+] 0.3s
conda-forge/linux-64  19%
conda-forge/noarch    35%[+] 0.4s
conda-forge/linux-64  28%
conda-forge/noarch    54%[+] 0.5s
conda-forge/linux-64  37%
conda-forge/noarch    74%[+] 0.6s
conda-forge/linux-64  47%
conda-forge/noarch    92%[+] 0.7s
conda-forge/linux-64  47%
conda-forge/noarch    92%conda-forge/noarch                                
[+] 0.8s
conda-forge/linux-64  60%[+] 0.9s
conda-forge/linux-64  77%[+] 1.0s
conda-forge/linux-64  95%[+] 1.1s
conda-forge/linux-64  95%[+] 1.2s
conda-forge/linux-64  95%conda-forge/linux-64                              

Pinned packages:

  - python=3.13

Pinned packages:

  - python_abi[version="=3.13",build="*cp313*"]

Pinned packages:

  - cuda-version=13

Pinned packages:

  - python=3.13


Transaction

  Prefix: /usr/local

  Updating specs:

   - openmm
   - openmmforcefields
   - parmed
   - pdb2pqr
   - pd

Protonate the enzyme at pH 7.0 using pdb2pqr using the PROPKA method

In [3]:
# purpose: set up an md simulation with a glycan ligand (force field: GLYCAM-06)
# first, protonate the protein using pdb2pqr and propka
import os
import sys
print("Protonating enzyme at pH 7.0...")
os.system("pdb2pqr --ff=AMBER --titration-state-method=propka --with-ph=7.0 OsChia1b-dct_relax.pdb --ffout=AMBER OsChia1b-dct_relax_pH7.pdb")

Protonating enzyme at pH 7.0...


0

Import dependencies and begin setting up the system. Check disulfide bonds and add any if needed

In [30]:
import openmm as mm
import openmm.app as app
import openmm.unit as unit
import parmed as pmd

print("Building OpenMM system...")
enzyme_pdb = app.PDBFile('OsChia1b-dct_relax_pH7.pdb')
ligand_pdb = app.PDBFile('GlcNAc5_top_OsChia1b-dct_restored.pdb')

modeller = app.Modeller(enzyme_pdb.topology, enzyme_pdb.positions)
modeller.add(ligand_pdb.topology, ligand_pdb.positions)

print("Existing Disulfide Bonds in Topology:")
for bond in modeller.topology.bonds():
    atom1, atom2 = bond
    if atom1.name == 'SG' and atom2.name == 'SG':
        print(f"  Bonded: {atom1.residue.name}{atom1.residue.id} to {atom2.residue.name}{atom2.residue.id}")

n_bonds_before = __builtins__.sum(1 for _ in modeller.topology.bonds())
modeller.topology.createDisulfideBonds(modeller.positions)
n_bonds_after = __builtins__.sum(1 for _ in modeller.topology.bonds())
print(f"Disulfide bonds added: {n_bonds_after - n_bonds_before}")

Building OpenMM system...
Existing Disulfide Bonds in Topology:
  Bonded: CYS18 to CYS3
  Bonded: CYS24 to CYS12
  Bonded: CYS31 to CYS17
  Bonded: CYS39 to CYS35
  Bonded: CYS42 to CYS15
  Bonded: CYS140 to CYS78
  Bonded: CYS160 to CYS152
  Bonded: CYS291 to CYS259
Disulfide bonds added: 0


Load force fields, add hydrogens to the ligand, and verify the protonation states of both His and Glu

In [31]:
forcefield = app.ForceField('amber19-all.xml', 'amber14/GLYCAM_06j-1.xml', 'amber19/opc3.xml')

ligand_residue_names = {'ROH', '4YB', '0YB'}
ligand_residues = [r for r in modeller.topology.residues() if r.name in ligand_residue_names]

print(f"Found {len(ligand_residues)} ligand residues for hydrogen addition: {[(r.chain.id, r.name) for r in ligand_residues]}")
assert len(ligand_residues) > 0, "No ligand residues found — check chain IDs"

app.Modeller.loadHydrogenDefinitions('glycam-hydrogens.xml')
modeller.addHydrogens(forcefield)

print("--- His/Glu states of enzyme after addHydrogens ---")
for res in modeller.topology.residues():
    if res.name in ('HIS', 'HID', 'HIE', 'HIP', 'GLU', 'GLH'):
        atom_names = [a.name for a in res.atoms()]
        print(res.name, res.id, atom_names)

Found 6 ligand residues for hydrogen addition: [(' ', 'ROH'), (' ', '4YB'), (' ', '4YB'), (' ', '4YB'), (' ', '4YB'), (' ', '0YB')]
--- His/Glu states of enzyme after addHydrogens ---
GLU 1 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3', 'H2', 'H3']
GLU 67 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
HIS 72 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'H', 'HA', 'HB2', 'HB3', 'CD2', 'ND1', 'CE1', 'NE2', 'HD1', 'HD2', 'HE1', 'HE2']
GLU 87 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 106 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
GLU 111 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H', 'HA', 'HB2', 'HB3', 'HG2', 'HG3']
HIS 121 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD2', 'ND1', 'CE1', 'NE2', 'H', 'HA', 'HB2', 'HB3', 'HD1', 'HD2', 'HE1']
GLU 122 ['N', 'CA', 'C', 'CB', 'O', 'CG', 'CD', 'OE1', 'OE2', 'H'

Add solvent, neutralize with 0.15 M NaCl, officially create the system with constraints on hydrogen bonds, and export .prmtop, .pdb, and .cif files for further downstream analysis like MM/PBSA

Note that compared to my previous code I had to use unit.nanometer, unit.molar, app.HBonds etc since I did not import openmm as * (safer not to do it that way, as it caused conflicts)

In [34]:
# important: if you get an error, do NOT rerun this cell by itself, it will start adding even more waters
# go back to the import openmm as mm cell and remake the topology or else you may end up with a box inside of a box
modeller.addSolvent(forcefield, model='tip3p', boxShape='octahedron', padding=1.1*unit.nanometers, positiveIon='Na+', negativeIon='Cl-', ionicStrength=0.15*unit.molar, neutralize=True)

system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints=app.HBonds
)

system_for_export = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints=None,
    rigidWater=False
)

from openmm.app import PDBFile
PDBFile.writeFile(modeller.topology, modeller.positions, open('solvated_system.pdb', 'w'))
from openmm.app import PDBxFile
PDBxFile.writeFile(modeller.topology, modeller.positions, open('solvated_system.cif', 'w'))
structure = pmd.openmm.load_topology(modeller.topology, system_for_export, modeller.positions)
structure.save('solvated_system.prmtop', format='amber')
structure.save('solvated_system.inpcrd', format='rst7')
print(".prmtop file created.")

.prmtop file created.


After solvation, count protein, ligand, and water molecules/atoms to ensure the system was properly formed/solvated

In [35]:
# residue names based on AMBER/GLYCAM06 conventions
protein_residues = {
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLU', 'GLN', 'GLY', 'HIS', 'ILE', 'LEU',
    'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'HID', 'HIE',
    'HIP', 'CYX', 'ASH', 'GLH', 'LYN'
}
ligand_residues = {'ROH', '0YB', '4YB'}
water_residues = {'HOH'}
sodium = {'NA'}
chloride = {'CL'}

# create variables for counts
num_protein_residues = 0
num_protein_atoms = 0
num_ligand_residues = 0
num_ligand_atoms = 0
num_water_molecules = 0
num_water_atoms = 0
num_sodium_atoms = 0
num_chloride_atoms = 0
num_other_atoms = 0

# count residues and atoms
for res in modeller.topology.residues():
    name = res.name.upper()
    num_atoms = len(list(res.atoms()))

    if name in protein_residues:
        num_protein_residues += 1
        num_protein_atoms += num_atoms
    elif name in ligand_residues:
        num_ligand_residues += 1
        num_ligand_atoms += num_atoms
    elif name in water_residues:
        num_water_molecules += 1
        num_water_atoms += num_atoms
    elif name in sodium:
        num_sodium_atoms += num_atoms
    elif name in chloride:
        num_chloride_atoms += num_atoms
    else:
        num_other_atoms += num_atoms

print("Solvated System Summary:")
print(f"Protein: {num_protein_residues} residues, {num_protein_atoms} atoms")
print(f"Ligand:  {num_ligand_residues} residues, {num_ligand_atoms} atoms")
print(f"Total water molecules: {num_water_molecules} ({num_water_atoms} atoms)")
print(f"Ions: {num_sodium_atoms} Na+ ions, {num_chloride_atoms} Cl- ions")
print(f"Other atoms: {num_other_atoms} atoms")
print(f"Total atoms: {modeller.topology.getNumAtoms()}")

Solvated System Summary:
Protein: 299 residues, 4283 atoms
Ligand:  6 residues, 138 atoms
Total water molecules: 21426 (64278 atoms)
Ions: 58 Na+ ions, 58 Cl- ions
Other atoms: 0 atoms
Total atoms: 68815


Equilibration phases with restraints, gradual heating under NVT (400 ps), then gradual release of restraints under NPT (1 ns). Note since this is run in Google Colab, the platform is CUDA and CudaPrecision

In [36]:
import openmm as mm
import openmm.app as app
import openmm.unit as unit

# apply heavy atom restraints which will be gradually released later
print("Step 1: Applying positional restraints before equilibration...")
restraint = mm.CustomExternalForce('k*periodicdistance(x, y, z, x0, y0, z0)^2')
restraint.addGlobalParameter('k', 100.0 * unit.kilocalories_per_mole/unit.angstrom**2)
restraint.addPerParticleParameter('x0')
restraint.addPerParticleParameter('y0')
restraint.addPerParticleParameter('z0')

restrained_atom_indices = []
for atom in modeller.topology.atoms():
    if atom.residue.name in ('HOH', 'WAT', 'NA', 'CL', 'K'):
        continue
    if atom.element is not None and atom.element.symbol != 'H':
        restraint.addParticle(atom.index, modeller.positions[atom.index])
        restrained_atom_indices.append(atom.index)

system.addForce(restraint)
print(f"Restrained {len(restrained_atom_indices)} heavy atoms.")

# initialize integrator with random seed
integrator = mm.LangevinMiddleIntegrator(0*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
seed = 13
integrator.setRandomNumberSeed(seed)
print(f"Random seed: {seed}")

# using openCL instead of CUDA due to compatibility issues within colab
platform = mm.Platform.getPlatformByName('OpenCL')
properties = {'OpenCLPrecision': 'mixed'}

simulation = app.Simulation(modeller.topology, system, integrator, platform, properties)
simulation.context.setPositions(modeller.positions)

# energy minimization step
print("Step 2: Energy minimization and positional restraints...")
state_before = simulation.context.getState(getEnergy=True)
print(f"Potential energy before minimization: {state_before.getPotentialEnergy()}")
simulation.minimizeEnergy(maxIterations=10000)
state_after = simulation.context.getState(getEnergy=True)
print(f"Potential energy after minimization: {state_after.getPotentialEnergy()}")

# moving the system from its crystallized structure to 300 K abruptly can cause unfolding or sudden unwanted shifts in structure, so this allows the system to settle more gently.
print("Step 3: Gradual heating phase (0 K -> 300 K over 400 ps)...")
heating_steps = 100
steps_per_increment = 2000  # 2000 steps * 2 fs = 4 ps per temperature step, allows the system to slowly heat up
for i in range(heating_steps):
    current_temp = (i / heating_steps) * 300 * unit.kelvin
    integrator.setTemperature(current_temp)
    simulation.step(steps_per_increment)
    if i % 10 == 0:
        print(f"  -> Temperature reached: {current_temp.value_in_unit(unit.kelvin):.1f} K")
integrator.setTemperature(300*unit.kelvin)

# add barostat for NPT equilibration with gradual release of restraints over 1 ns
system.addForce(mm.MonteCarloBarostat(1.0*unit.bar, 300*unit.kelvin))
simulation.context.reinitialize(preserveState=True)

print("Step 4: Releasing positional restraints gradually...")
release_steps = 10
initial_k = 100.0
for i in range(release_steps):
    scale = 1.0 - (i / (release_steps - 1))  # 1.0 -> 0.0
    simulation.context.setParameter('k', initial_k * scale * unit.kilocalories_per_mole/unit.angstrom**2)
    simulation.step(50000)  # 100 ps per stage at 2 fs/step
    print(f"  -> Restraint scale: {scale:.2f}")

Step 1: Applying positional restraints before equilibration...
Restrained 2305 heavy atoms.
Random seed: 13
Step 2: Energy minimization and positional restraints...
Potential energy before minimization: -729131.4598546615 kJ/mol
Potential energy after minimization: -1321862.752801747 kJ/mol
Step 3: Gradual heating phase (0 K -> 300 K over 400 ps)...
  -> Temperature reached: 0.0 K
  -> Temperature reached: 30.0 K
  -> Temperature reached: 60.0 K
  -> Temperature reached: 90.0 K
  -> Temperature reached: 120.0 K
  -> Temperature reached: 150.0 K
  -> Temperature reached: 180.0 K
  -> Temperature reached: 210.0 K
  -> Temperature reached: 240.0 K
  -> Temperature reached: 270.0 K
Step 4: Releasing positional restraints gradually...
  -> Restraint scale: 1.00
  -> Restraint scale: 0.89
  -> Restraint scale: 0.78
  -> Restraint scale: 0.67
  -> Restraint scale: 0.56
  -> Restraint scale: 0.44
  -> Restraint scale: 0.33
  -> Restraint scale: 0.22
  -> Restraint scale: 0.11
  -> Restraint sc

Production run for MD, saving every 20 ps, total runtime is 100 ns

In [37]:
simulation.reporters.append(app.DCDReporter('trajectory.dcd', 10000))
simulation.reporters.append(app.CheckpointReporter('checkpoint.chk', 10000))

# report to save to log file
simulation.reporters.append(app.StateDataReporter(
    'OsChia1b_G5_seed13.log', 10000, step=True, potentialEnergy=True,
    temperature=True, progress=True, totalSteps=50000000, speed=True, remainingTime=True
))

# reporter to console
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 10000, step=True, potentialEnergy=True,
    temperature=True, progress=True, totalSteps=50000000, speed=True, remainingTime=True
))

simulation.step(50000000)
print("Simulation complete.")

Streaming output truncated to the last 5000 lines.
1.4%,720000,-1133051.3010556838,299.39154464074005,187,12:38:51
1.5%,730000,-1133096.5383778336,299.6814260670314,187,12:38:56
1.5%,740000,-1133378.1525169145,299.29691406303374,187,12:38:20
1.5%,750000,-1134590.2714365376,298.7700783265739,187,12:39:21
1.5%,760000,-1135170.9887561426,298.87916925764563,187,12:37:51
1.5%,770000,-1132968.7413584031,300.2570035830013,187,12:37:33
1.6%,780000,-1135049.077303106,300.0589146887453,187,12:38:19
1.6%,790000,-1133811.3065693965,301.1658676781871,187,12:37:48
1.6%,800000,-1132696.8560883747,301.39586097634066,187,12:37:45
1.6%,810000,-1132778.392028086,300.13834474106943,187,12:37:16
1.6%,820000,-1133660.61639803,300.938763922988,187,12:37:19
1.7%,830000,-1131904.0463659642,299.21682497644343,187,12:36:41
1.7%,840000,-1133546.3143083313,301.45425294177846,187,12:36:35
1.7%,850000,-1134391.1098137992,299.7576659753943,187,12:36:59
1.7%,860000,-1134688.450058654,300.43207708008435,187,12:36:51
1.

In [39]:
import os
import shutil

# define folder to be saved to
gdrive_dest_dir = '/content/drive/MyDrive/MDColab/OsChia1b/holo'
os.makedirs(gdrive_dest_dir, exist_ok=True)

# save all outputs including log
files_to_copy = [
    'solvated_system.pdb',
    'solvated_system.cif',
    'solvated_system.prmtop',
    'solvated_system.inpcrd',
    'trajectory.dcd',
    'checkpoint.chk',
    'OsChia1b_G5_seed13.log'
]

# copy files to target directory
for file_name in files_to_copy:
    if os.path.exists(file_name):
        print(f"Copying {file_name} to Google Drive...")
        shutil.copy(file_name, os.path.join(gdrive_dest_dir, file_name))
    else:
        print(f"Warning: {file_name} not found.")

print("Finished saving files to Google Drive.")

Copying solvated_system.pdb to Google Drive...
Copying solvated_system.cif to Google Drive...
Copying solvated_system.prmtop to Google Drive...
Copying solvated_system.inpcrd to Google Drive...
Copying trajectory.dcd to Google Drive...
Copying checkpoint.chk to Google Drive...
Copying OsChia1b_G5_seed13.log to Google Drive...
Finished saving files to Google Drive.
